In [ ]:
text="Describe this image in detail."
image=None  # set to an image path (e.g. "./photo.jpg") for image + text
ExecutionProvider="NvTensorRTRTXExecutionProvider"
model_folder = "./model"

In [ ]:
from winml import register_execution_providers
register_execution_providers(ExecutionProvider)

In [ ]:
import json
import time

import onnxruntime_genai as og

# Load the model and the multimodal processor
model = og.Model(model_folder)
processor = model.create_multimodal_processor()
tokenizer = og.Tokenizer(model)
tokenizer_stream = processor.create_stream()

# Build the chat prompt (optionally with an image)
images = None
if image:
    images = og.Images.open(image)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": text},
            ],
        }
    ]
else:
    messages = [{"role": "user", "content": text}]

full_prompt = tokenizer.apply_chat_template(json.dumps(messages), add_generation_prompt=True)
inputs = processor(full_prompt, images=images)

# Create params and generator
params = og.GeneratorParams(model)
params.set_search_options(max_length=4096)

generator = og.Generator(model, params)
generator.set_inputs(inputs)

print("")
print("Output: ", end="", flush=True)

token_times = []

# Stream the output
while not generator.is_done():
    start_time = time.time()
    generator.generate_next_token()
    new_token = generator.get_next_tokens()[0]
    token_times.append(time.time() - start_time)

    print(tokenizer_stream.decode(new_token), end="", flush=True)

print()

# Calculate and display timing statistics
if token_times:
    total_tokens = len(token_times)
    avg_time = sum(token_times) / total_tokens

    print(f"Total tokens generated: {total_tokens}")
    print(f"Average time per token: {avg_time:.4f} seconds")
    print(f"Tokens per second: {total_tokens / sum(token_times):.2f}")

del generator